# Data Extracting Pipeline



In [2]:
import pandas as pd
import json
import os

## 1. Setup and File Discovery

In [3]:
base_path = "C:/Users/Art/Downloads/ScopusData/ScopusData2010-2023"

## 2. Load and Parse Data

In [4]:
extracted_data = []
years = range(2010, 2024)
for year in years:
    year_folder = os.path.join(base_path, str(year))
    
    if not os.path.exists(year_folder):
        print(f"Skipping {year} (Folder not found)")
        continue

    print(f"Processing year: {year}...")
    for filename in os.listdir(year_folder):
        file_path = os.path.join(year_folder, filename)

        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except:
            continue

        # Default
        title = "Unknown"
        publication_name = "Unknown"
        authors_list = []
        abstract = "No Abstract"
        keywords_str = "-"
        scopus_link = "No Link"
        subject_codes_list = []

        try:
            root = data.get('abstracts-retrieval-response', {})
            item = root.get('item', {})
            
            bibrecord = item.get('bibrecord', {})
            head = bibrecord.get('head', {})
            coredata = root.get('coredata', {})

            # Title
            title = coredata.get('dc:title') or \
                    head.get('citation-title') or \
                    head.get('source', {}).get('title') or \
                    "Unknown Title"

            # Publication Name
            publication_name = coredata.get('prism:publicationName') or \
                               head.get('source', {}).get('sourcetitle') or \
                               "Unknown Source"

            # Authors
            author_group = head.get('author-group', [])
            if isinstance(author_group, dict):
                author_group = [author_group]

            for group in author_group:
                authors = group.get('author', [])
                if isinstance(authors, dict):
                    authors = [authors]

                for auth in authors:
                    given = auth.get('ce:given-name', '') or auth.get('given-name', '')
                    surname = auth.get('ce:surname', '') or auth.get('surname', '')
                    indexed_name = auth.get('ce:indexed-name') or auth.get('indexed-name')

                    full_name = f"{given} {surname}".strip()
                    if not full_name and indexed_name:
                        full_name = indexed_name
                    if full_name:
                        authors_list.append(full_name)

            
            if not authors_list:
                dc_creator = coredata.get('dc:creator', {})
                dc_authors = dc_creator.get('author', []) if isinstance(dc_creator, dict) else []
                if isinstance(dc_authors, dict):
                    dc_authors = [dc_authors]

                for a in dc_authors:
                    given = a.get('ce:given-name', '') or a.get('given-name', '')
                    surname = a.get('ce:surname', '') or a.get('surname', '')
                    indexed_name = a.get('ce:indexed-name') or a.get('indexed-name')

                    full_name = f"{given} {surname}".strip()
                    if not full_name and indexed_name:
                        full_name = indexed_name
                    if full_name:
                        authors_list.append(full_name)

            authors_str = ", ".join(authors_list)

            # Abstract
            abstract = coredata.get('dc:description', "No Abstract")

            # Keywords
            keywords_raw = root.get('authkeywords', {}).get('author-keyword', [])
            if isinstance(keywords_raw, dict):
                keywords_raw = [keywords_raw]

            keywords = [kw.get('$') for kw in keywords_raw if isinstance(kw, dict) and kw.get('$')]
            keywords_str = ", ".join(keywords)

            # Link
            try:
                src = head.get('source', {})
                website = src.get('website', {})
                addr = website.get('ce:e-address', {})

                if isinstance(addr, dict):
                    scopus_link = addr.get('$')

                if not scopus_link:
                    link_list = coredata.get("link", [])
                    if isinstance(link_list, dict):
                        link_list = [link_list]
                    if isinstance(link_list, list):
                        for ln in link_list:
                            if isinstance(ln, dict) and ln.get("@ref") == "scopus":
                                scopus_link = ln.get("@href")
                                break

                if not scopus_link:
                    scopus_link = "No Link"
            except:
                pass
            
            #Scopus Field Codes
            subject_codes = set()
            try:
                subj_raw = root.get("subject-areas", {}).get("subject-area", [])
                if isinstance(subj_raw, dict):
                    subj_raw = [subj_raw]

                for subj in subj_raw:
                    code = subj.get("@code")
                    if code and len(code) >= 2:
                        subject_codes.add(code[:2])
            except:
                pass

            subject_codes_list = sorted(list(subject_codes))
            
        except:
            scopus_link = "No Link"

        extracted_data.append({
            'Title': title,
            'Publication Name': publication_name,
            'Date': coredata.get('prism:coverDate') if 'coredata' in locals() else None,
            'Authors': ", ".join(authors_list),
            'Author_Count': len(authors_list),
            'Keywords': keywords_str,
            'Abstract': abstract,
            'Link': scopus_link,
            'Source_File': os.path.basename(filename),
            'Subject_Codes': subject_codes_list
        })

print(f"Successfully parsed {len(extracted_data)} records.")


Processing year: 2010...
Processing year: 2011...
Processing year: 2012...
Processing year: 2013...
Processing year: 2014...
Processing year: 2015...
Processing year: 2016...
Processing year: 2017...
Processing year: 2018...
Processing year: 2019...
Processing year: 2020...
Processing year: 2021...
Processing year: 2022...
Processing year: 2023...
Successfully parsed 36475 records.


In [5]:
fieldcode = { 
    10: 'Multidisciplinary',
    11: 'Agri & Bio',
    12: 'Arts & Hum',
    13: 'Biochem & Genetics',
    14: 'Business & Mgmt',
    15: 'Chem Eng',
    16: 'Chemistry',
    17: 'Comp Sci',
    18: 'Decision Sci',
    19: 'Earth & Planet',
    20: 'Econ & Finance',
    21: 'Energy',
    22: 'Engineering',
    23: 'Environment',
    24: 'Immuno & Micro',
    25: 'Materials',
    26: 'Math',
    27: 'Medicine',
    28: 'Neuroscience',
    29: 'Nursing',
    30: 'Pharma & Tox',
    31: 'Physics',
    32: 'Psychology',
    33: 'Social Sci',
    34: 'Vet Med',
    35: 'Dentistry',
    36: 'Health'
}

for record in extracted_data:
    codes = record.get('Subject_Codes', [])
    
    converted = []
    for code in codes:
        try:
            num = int(code)
            if num in fieldcode:
                converted.append(fieldcode[num])
        except:
            pass
    
    record['Subject_Fields'] = converted


## 3. Create DataFrame and Transform

In [6]:
df = pd.DataFrame(extracted_data)
df = df.drop_duplicates(subset=["Title"], keep="first")
df = df.drop(columns=["Source_File", "Subject_Codes"], errors="ignore")

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

df['Year'] = df['Date'].dt.year

df['Publication Name'] = df['Publication Name'].fillna('Unknown')
df['Keywords'] = df['Keywords'].fillna('')
df['Abstract'] = df['Abstract'].fillna('')
df['Link'] = df['Link'].fillna('No Link')


display(df.head())
df.info()


,Title,Publication Name,Date,Authors,Author_Count,Keywords,Abstract,Link,Subject_Fields,Year
0,Modelling of tubular-designed solid oxide fuel...,Journal of Power Sources,2010-01-01,"Dokmaingam, Laosiripojana, Assabumrungrat, Soo...",4,"Biogas, Ethanol, Indirect internal reforming, ...",Mathematical models of an indirect internal re...,No Link,"[Chemistry, Energy, Engineering]",2010
1,Purification and concentration of a rhamnolipi...,Bioresource Technology,2010-01-01,"Thitima Sarachat, Orathai Pornsunthorntawee, S...",6,"Biosurfactants, Foam fractionation, Pseudomona...",Pseudomonas aeruginosa SP4 was cultivated to p...,No Link,"[Chem Eng, Energy, Environment]",2010
2,Sequential deposition of polydisperse particle...,Physica A: Statistical Mechanics and its Appli...,2010-01-01,"Panu Danwanichakul, Tawatchai Charinpanitkul",2,"Colloid, Deposition, Double layer interaction,...",Adsorption of charged colloidal particles to o...,http://www.journals.elsevier.com/physica-a-sta...,"[Math, Physics]",2010
3,Leverage-reducing exchange offers and bondhold...,International Review of Economics and Finance,2010-01-01,"Pattanaporn Kitsabunnarat-Chatjuthamard, Peter...",4,"Capital structure, Exchange offers, Wealth tra...",This paper re-examines the intersecurity wealt...,No Link,[Econ & Finance],2010
4,Expression of immune-related genes in the dige...,Developmental and Comparative Immunology,2010-01-01,"Wipasiri Soonthornchai, Padermsak Jarayabhand,...",7,"Antimicrobial peptides, C-type lectin, Innate ...","In all previous studies, to study shrimp immun...",https://www.sciencedirect.com/science/journal/...,"[Biochem & Genetics, Immuno & Micro]",2010


<class 'pandas.core.frame.DataFrame'>
Index: 36163 entries, 0 to 36474
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Title             36163 non-null  object        
 1   Publication Name  36163 non-null  object        
 2   Date              36163 non-null  datetime64[ns]
 3   Authors           36163 non-null  object        
 4   Author_Count      36163 non-null  int64         
 5   Keywords          36163 non-null  object        
 6   Abstract          36163 non-null  object        
 7   Link              36163 non-null  object        
 8   Subject_Fields    36163 non-null  object        
 9   Year              36163 non-null  int32         
dtypes: datetime64[ns](1), int32(1), int64(1), object(7)
memory usage: 2.9+ MB


## 4. Analysis and Save

In [7]:
# Publications per Year
pub_per_year = df['Year'].value_counts().sort_index()
print("\n-Publications per Year ")
print(pub_per_year)

# Save File
output_filename = 'info.csv'
df.to_csv(output_filename, index=False)
print(f"\nSaved processed data to {output_filename}")


-Publications per Year 
Year
2010    1671
2011    1717
2012    1835
2013    1927
2014    2063
2015    2049
2016    2337
2017    2462
2018    2745
2019    3068
2020    3380
2021    3794
2022    4232
2023    2883
Name: count, dtype: int64

Saved processed data to info.csv
